In [13]:
!pip install nibabel scikit-learn -q

import os, warnings, zipfile, glob
from typing import List

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision.transforms as T
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

print("Import done.")

Import done.


In [14]:
NUM_CLIENTS = 3
FL_ALPHA    = 0.5
FL_BETA     = 1.0
IMG_SIZE    = 224
VAL_FRAC    = 0.1

CSV_PATH = '/kaggle/input/datasets/calamaridino/mri-skull-stripped/skull_stripped/participants.tsv'
IMG_DIR  = '/kaggle/input/datasets/calamaridino/mri-skull-stripped/skull_stripped/'

OUTPUT_DIR = '/kaggle/working/federated_split'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SCANNER_NAMES = {0: 'Siemens', 1: 'Philips', 2: 'GE'}

print(f"Output dir: {OUTPUT_DIR}")

Output dir: /kaggle/working/federated_split


In [15]:
def scanner_siemens(x: torch.Tensor) -> torch.Tensor:
    """Tăng độ sáng (high-field contrast)."""
    return torch.clamp(x * 1.25, 0.0, 1.0)

def scanner_philips(x: torch.Tensor) -> torch.Tensor:
    """Thêm Gaussian noise (thermal noise đặc trưng Philips)."""
    return torch.clamp(x + torch.randn_like(x) * 0.04, 0.0, 1.0)

def scanner_ge(x: torch.Tensor) -> torch.Tensor:
    """Làm mờ nhẹ (lower sharpness GE)."""
    return F.avg_pool2d(x.unsqueeze(0), 3, 1, 1).squeeze(0)

SCANNER_TRANSFORMS = {
    0: scanner_siemens,
    1: scanner_philips,
    2: scanner_ge,
}

print("Scanner transforms defined:")
for cid, fn in SCANNER_TRANSFORMS.items():
    print(f"  Client {cid} ({SCANNER_NAMES[cid]}): {fn.__name__}")

Scanner transforms defined:
  Client 0 (Siemens): scanner_siemens
  Client 1 (Philips): scanner_philips
  Client 2 (GE): scanner_ge


In [16]:
def nifti_to_3channel(img_path: str, size: int = IMG_SIZE) -> torch.Tensor:
    """Load NIfTI, lấy 3 slice trực giao, Z-score normalize, stack thành tensor 3 kênh."""
    vol        = nib.load(img_path).get_fdata().astype(np.float32)
    sx, sy, sz = vol.shape
    slices     = [
        vol[sx // 2, :, :],
        vol[:, sy // 2, :],
        vol[:, :, sz // 2],
    ]
    tensors = []
    for sl in slices:
        mean, std = sl.mean(), sl.std()
        sl = (sl - mean) / (std + 1e-8)
        sl = np.clip(sl, -3.0, 3.0)
        sl = (sl + 3.0) / 6.0
        pil = Image.fromarray((sl * 255).astype(np.uint8)).resize(
            (size, size), Image.BILINEAR
        )
        tensors.append(T.ToTensor()(pil))
    return torch.cat(tensors, dim=0)   # shape: (3, H, W)


class BrainAgeDataset(Dataset):
    """Dataset gốc — KHÔNG áp scanner transform ở đây."""
    def __init__(self, csv_file: str, img_dir: str):
        self.metadata = pd.read_csv(csv_file, sep='\t', low_memory=False)
        self.img_dir  = img_dir
        self.metadata = self.metadata.dropna(subset=['AGE_AT_SCAN']).reset_index(drop=True)
        print(f"  Total subjects in metadata: {len(self.metadata)}")

    def __len__(self):
        return len(self.metadata)

    def _find_path(self, sub_id: str):
        candidates = [
            os.path.join(self.img_dir, sub_id, 'anat',
                         f"{sub_id}_T1w.nii", f"{sub_id}_T1w.nii"),
            os.path.join(self.img_dir, sub_id, 'anat', f"{sub_id}_T1w.nii"),
            os.path.join(self.img_dir, sub_id, f"{sub_id}_T1w.nii"),
        ]
        for p in candidates:
            if os.path.exists(p) and os.path.getsize(p) > 0:
                return p
        return None

    def __getitem__(self, idx):
        row  = self.metadata.iloc[idx]
        age  = float(row['AGE_AT_SCAN'])
        path = self._find_path(row['subject_id'])
        if path is None:
            return self.__getitem__((idx + 1) % len(self))
        try:
            return nifti_to_3channel(path), torch.tensor(age, dtype=torch.float32)
        except Exception:
            return self.__getitem__((idx + 1) % len(self))


print("BrainAgeDataset ready.")

BrainAgeDataset ready.


In [17]:
class ClientSubset(Dataset):
    """
    Wrapper áp đúng scanner transform cho từng client.
    Nhận vào base Dataset và danh sách global index.
    """
    def __init__(self, base_dataset: BrainAgeDataset,
                 indices: np.ndarray,
                 client_id: int):
        self.base      = base_dataset
        self.indices   = indices
        self.transform = SCANNER_TRANSFORMS[client_id]
        self.scanner   = SCANNER_NAMES[client_id]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = self.indices[idx]
        x, y = self.base[global_idx]
        x    = self.transform(x)      # ← feature shift áp tại đây
        return x, y


print("ClientSubset ready.")
print("Usage: ClientSubset(full_dataset, global_indices, client_id=0)")

ClientSubset ready.
Usage: ClientSubset(full_dataset, global_indices, client_id=0)


In [18]:
print("Loading dataset …")
full_dataset = BrainAgeDataset(CSV_PATH, IMG_DIR)
print(f"Total samples: {len(full_dataset)}")

train_idx, test_idx = train_test_split(
    np.arange(len(full_dataset)),
    test_size=0.2,
    random_state=42,
)

print(f"Train: {len(train_idx)}  |  Test: {len(test_idx)}")

Loading dataset …
  Total subjects in metadata: 1112
Total samples: 1112
Train: 889  |  Test: 223


In [19]:
class FederatedMedicalPartitioner:
    """
    Chia train_idx thành num_clients phần theo Dirichlet:
      - beta  → quantity skew (số lượng mẫu mỗi client)
      - alpha → label skew (phân phối tuổi trong từng age-bin)
    """
    def __init__(self, full_dataset: BrainAgeDataset,
                 train_idx: np.ndarray,
                 num_clients: int = 3,
                 alpha: float = 0.5,
                 beta: float  = 1.0,
                 seed: int    = 42):
        self.base        = full_dataset
        self.train_idx   = train_idx
        self.num_clients = num_clients
        self.alpha       = alpha
        self.beta        = beta
        np.random.seed(seed)

        self.labels         = self._extract_labels()
        self.client_indices = self._partition()   # list of np.ndarray (positions in train_idx)
        self._print_stats()

    def _extract_labels(self) -> np.ndarray:
        return np.array([
            float(self.base.metadata.iloc[i]['AGE_AT_SCAN'])
            for i in self.train_idx
        ])

    def _partition(self) -> List[np.ndarray]:
        labels = self.labels
        C      = self.num_clients

        q_proportions = np.random.dirichlet([self.beta] * C)

        bins            = np.linspace(labels.min(), labels.max(), 11)
        bin_assignments = np.digitize(labels, bins) - 1

        client_lists = [[] for _ in range(C)]

        for b in range(10):
            pos_in_bin = np.where(bin_assignments == b)[0]
            if len(pos_in_bin) == 0:
                continue
            np.random.shuffle(pos_in_bin)

            label_props = np.random.dirichlet([self.alpha] * C)
            combined    = label_props * q_proportions
            combined   /= combined.sum()

            cuts = (np.cumsum(combined[:-1]) * len(pos_in_bin)).astype(int)
            for c, chunk in enumerate(np.split(pos_in_bin, cuts)):
                client_lists[c].extend(chunk.tolist())

        return [np.array(lst) for lst in client_lists]

    def _print_stats(self):
        total = sum(len(idx) for idx in self.client_indices)
        print(f"\n{'─'*70}")
        print(f"  Non-IID Partition  (α={self.alpha}, β={self.beta})")
        print(f"{'─'*70}")
        print(f"  {'Client':<14} {'N':>6}  {'%':>6}  "
              f"{'Min':>7}  {'Max':>7}  {'Mean':>8}  Scanner   Transform")
        for c in range(self.num_clients):
            pos = self.client_indices[c]
            if len(pos) == 0:
                print(f"  Client {c:<8}  {'0':>6}  (empty)")
                continue
            ages = self.labels[pos]
            pct  = 100 * len(pos) / total
            fn   = SCANNER_TRANSFORMS[c].__name__
            print(f"  Client {c:<8}  {len(pos):>6}  {pct:>5.1f}%  "
                  f"{ages.min():>7.1f}  {ages.max():>7.1f}  "
                  f"{ages.mean():>8.1f}  {SCANNER_NAMES[c]:<9} {fn}")
        print(f"  {'Total':<14}  {total:>6}")
        print(f"{'─'*70}\n")


partitioner = FederatedMedicalPartitioner(
    full_dataset, train_idx,
    num_clients=NUM_CLIENTS,
    alpha=FL_ALPHA,
    beta=FL_BETA,
)
print("Partition done.")


──────────────────────────────────────────────────────────────────────
  Non-IID Partition  (α=0.5, β=1.0)
──────────────────────────────────────────────────────────────────────
  Client              N       %      Min      Max      Mean  Scanner   Transform
  Client 0            192   21.6%      7.0     52.0      16.1  Siemens   scanner_siemens
  Client 1            613   69.0%      6.5     56.2      17.0  Philips   scanner_philips
  Client 2             83    9.3%      7.3     58.0      20.9  GE        scanner_ge
  Total              888
──────────────────────────────────────────────────────────────────────

Partition done.


In [20]:
client_global_splits = {}

for cid in range(NUM_CLIENTS):
    positions  = partitioner.client_indices[cid]
    global_idx = train_idx[positions]

    n      = len(global_idx)
    v_size = max(4, int(VAL_FRAC * n))
    t_size = n - v_size

    g    = torch.Generator().manual_seed(42 + cid)
    perm = torch.randperm(n, generator=g).numpy()

    client_global_splits[cid] = {
        "train": global_idx[perm[:t_size]],
        "val":   global_idx[perm[t_size:]],
    }

    print(f"  Client {cid} ({SCANNER_NAMES[cid]}): "
          f"{t_size} train | {v_size} val  "
          f"→ transform: {SCANNER_TRANSFORMS[cid].__name__}")

  Client 0 (Siemens): 173 train | 19 val  → transform: scanner_siemens
  Client 1 (Philips): 552 train | 61 val  → transform: scanner_philips
  Client 2 (GE): 75 train | 8 val  → transform: scanner_ge


In [21]:
base_meta = full_dataset.metadata

# Test set
pd.DataFrame({
    'global_idx': test_idx,
    'subject_id': base_meta.iloc[test_idx]['subject_id'].values,
    'age':        base_meta.iloc[test_idx]['AGE_AT_SCAN'].values.astype(float),
}).to_csv(f"{OUTPUT_DIR}/split_test.csv", index=False)
np.save(f"{OUTPUT_DIR}/split_test_global_idx.npy", test_idx)

# Train set toàn bộ
pd.DataFrame({
    'global_idx': train_idx,
    'subject_id': base_meta.iloc[train_idx]['subject_id'].values,
    'age':        base_meta.iloc[train_idx]['AGE_AT_SCAN'].values.astype(float),
}).to_csv(f"{OUTPUT_DIR}/split_train.csv", index=False)
np.save(f"{OUTPUT_DIR}/split_train_global_idx.npy", train_idx)

# Per-client
for cid in range(NUM_CLIENTS):
    for split_name, g_idx in client_global_splits[cid].items():
        pd.DataFrame({
            'global_idx': g_idx,
            'subject_id': base_meta.iloc[g_idx]['subject_id'].values,
            'age':        base_meta.iloc[g_idx]['AGE_AT_SCAN'].values.astype(float),
            'scanner':    SCANNER_NAMES[cid],
            'transform':  SCANNER_TRANSFORMS[cid].__name__,
        }).to_csv(f"{OUTPUT_DIR}/split_client{cid}_{split_name}.csv", index=False)
        np.save(f"{OUTPUT_DIR}/split_client{cid}_{split_name}_global_idx.npy", g_idx)

    print(f"  Saved client {cid} ({SCANNER_NAMES[cid]}): "
          f"{len(client_global_splits[cid]['train'])} train | "
          f"{len(client_global_splits[cid]['val'])} val")

print("\nAll index splits saved.")

  Saved client 0 (Siemens): 173 train | 19 val
  Saved client 1 (Philips): 552 train | 61 val
  Saved client 2 (GE): 75 train | 8 val

All index splits saved.


In [22]:
def save_transformed_tensors(base_dataset: BrainAgeDataset,
                              client_global_splits: dict,
                              test_idx: np.ndarray,
                              out_dir: str = OUTPUT_DIR):
    """
    Áp scanner transform ngay lúc lưu — chỉ chạy 1 lần.
    Người dùng sau chỉ cần torch.load(), không cần định nghĩa
    transform hay ClientSubset trong notebook training.
    """
    print("Saving transformed tensors …")

    # ── Test set (không transform) ──
    xs, ys = [], []
    for i, global_i in enumerate(test_idx):
        x, y = base_dataset[global_i]
        xs.append(x); ys.append(y)
        if (i + 1) % 50 == 0:
            print(f"  test: {i+1}/{len(test_idx)}", end='\r')
    torch.save({'X': torch.stack(xs), 'y': torch.stack(ys)},
               f"{out_dir}/tensors_test.pt")
    print(f"  ✓ tensors_test.pt          ({len(ys)} samples, no transform)")

    # ── Per-client train / val ──
    for cid in range(NUM_CLIENTS):
        transform = SCANNER_TRANSFORMS[cid]
        for split_name, g_idx in client_global_splits[cid].items():
            xs, ys = [], []
            for i, global_i in enumerate(g_idx):
                x, y = base_dataset[global_i]
                xs.append(transform(x))   # ← transform áp 1 lần duy nhất tại đây
                ys.append(y)
                if (i + 1) % 50 == 0:
                    print(f"  client{cid}_{split_name}: {i+1}/{len(g_idx)}", end='\r')
            fname = f"tensors_client{cid}_{split_name}.pt"
            torch.save({'X': torch.stack(xs), 'y': torch.stack(ys)},
                       f"{out_dir}/{fname}")
            print(f"  ✓ {fname:<40} ({len(ys)} samples, {SCANNER_NAMES[cid]})")

    print("\nAll .pt tensors saved.")


save_transformed_tensors(full_dataset, client_global_splits, test_idx)

Saving transformed tensors …
  ✓ tensors_test.pt          (223 samples, no transform)
  ✓ tensors_client0_train.pt                 (173 samples, Siemens)
  ✓ tensors_client0_val.pt                   (19 samples, Siemens)
  ✓ tensors_client1_train.pt                 (552 samples, Philips)
  ✓ tensors_client1_val.pt                   (61 samples, Philips)
  ✓ tensors_client2_train.pt                 (75 samples, GE)
  ✓ tensors_client2_val.pt                   (8 samples, GE)

All .pt tensors saved.


In [23]:
print("\n" + "="*65)
print("  VERIFICATION")
print("="*65)

all_train_val_ids = set()
total_tv = 0

for cid in range(NUM_CLIENTS):
    for split_name in ["train", "val"]:
        df = pd.read_csv(f"{OUTPUT_DIR}/split_client{cid}_{split_name}.csv")
        all_train_val_ids.update(df['subject_id'].tolist())
        total_tv += len(df)
        print(f"  Client {cid} {split_name:5s}: {len(df):5d} samples  "
              f"[{df['scanner'].iloc[0]} / {df['transform'].iloc[0]}]")

df_test  = pd.read_csv(f"{OUTPUT_DIR}/split_test.csv")
test_ids = set(df_test['subject_id'].tolist())
overlap  = all_train_val_ids & test_ids

print(f"\n  Test set      : {len(df_test):5d} samples")
print(f"  Train+Val sum : {total_tv:5d} samples")
print(f"  Grand total   : {total_tv + len(df_test):5d} samples")
print(f"  Full dataset  : {len(full_dataset):5d} samples")
print(f"\n  ✓ Train-Test overlap : {len(overlap)} subjects "
      f"{'← OK' if len(overlap) == 0 else '← WARNING!'}")
print("="*65)


  VERIFICATION
  Client 0 train:   173 samples  [Siemens / scanner_siemens]
  Client 0 val  :    19 samples  [Siemens / scanner_siemens]
  Client 1 train:   552 samples  [Philips / scanner_philips]
  Client 1 val  :    61 samples  [Philips / scanner_philips]
  Client 2 train:    75 samples  [GE / scanner_ge]
  Client 2 val  :     8 samples  [GE / scanner_ge]

  Test set      :   223 samples
  Train+Val sum :   888 samples
  Grand total   :  1111 samples
  Full dataset  :  1112 samples

  ✓ Train-Test overlap : 0 subjects ← OK


In [24]:
zip_path = '/kaggle/working/federated_brain_age_splits.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
        zf.write(fpath, arcname=os.path.basename(fpath))

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Created: {zip_path}  ({size_mb:.2f} MB)")

Created: /kaggle/working/federated_brain_age_splits.zip  (370.02 MB)


In [26]:
SPLIT_DIR = '/kaggle/input/datasets/phanhongdng/brainage-split'

def load_client_datasets_from_tensors(split_dir: str = SPLIT_DIR):
    result = {}

    data = torch.load(f"{split_dir}/tensors_test.pt")
    result['test'] = TensorDataset(data['X'], data['y'])
    print(f"  {'test':20s}: {len(result['test']):5d} samples  (no transform)")

    for cid in range(NUM_CLIENTS):
        for split_name in ["train", "val"]:
            key  = f"client{cid}_{split_name}"
            data = torch.load(f"{split_dir}/tensors_{key}.pt")
            result[key] = TensorDataset(data['X'], data['y'])
            print(f"  {key:20s}: {len(result[key]):5d} samples  [{SCANNER_NAMES[cid]}]")

    return result

# Ví dụ sử dụng:
datasets = load_client_datasets_from_tensors()

dl_c0_train = DataLoader(datasets['client0_train'], batch_size=8, shuffle=True)
x_batch, y_batch = next(iter(dl_c0_train))
print("Helper ready.")

  test                :   223 samples  (no transform)
  client0_train       :   173 samples  [Siemens]
  client0_val         :    19 samples  [Siemens]
  client1_train       :   552 samples  [Philips]
  client1_val         :    61 samples  [Philips]
  client2_train       :    75 samples  [GE]
  client2_val         :     8 samples  [GE]

Batch shape: torch.Size([8, 3, 224, 224])  |  Labels: tensor([18.6000, 55.4000, 18.6000, 22.8100])
Helper ready.
